# Transfer Learning for Trading

This notebook demonstrates how to use transfer learning to leverage pre-trained knowledge for new trading tasks.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from src.models.transfer_learner import TransferModel
from src.data.binance_fetcher import BinanceFetcher
from src.features.feature_engineering import FeatureEngineer

## Configuration

Load and configure transfer learning settings.

In [ ]:
# Load configuration
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Transfer learning configuration
transfer_config = {
    "layer_dims": [128, 64, 32],
    "head_dim": 16,
    "layer_config": {
        "batch_norm": True,
        "dropout": 0.1
    }
}

## Data Preparation

Fetch historical data and prepare tasks for transfer learning.

In [ ]:
# Initialize data fetcher
fetcher = BinanceFetcher(config['data']['binance'])

# Fetch historical data
data = await fetcher.fetch_historical_data()

# Initialize feature engineer
engineer = FeatureEngineer(
    input_file='data/raw/BTCUSDT_data.parquet',
    output_dir='data/processed',
    config=config
)

# Generate features
await engineer.generate_features()

## Task Creation

Create source and target tasks for transfer learning.

In [ ]:
def create_tasks(data, window_size=20):
    """Create source and target tasks."""
    # Calculate returns for price prediction (source task)
    returns = data['close'].pct_change()
    price_targets = returns.shift(-1)
    
    # Calculate volatility for volatility prediction (target task)
    volatility = returns.rolling(window=window_size).std()
    volatility_targets = volatility.shift(-1)
    
    # Create datasets
    features = data['features'].values
    source_targets = price_targets.values
    target_targets = volatility_targets.values
    
    # Remove NaN values
    valid_idx = ~(np.isnan(source_targets) | np.isnan(target_targets))
    features = features[valid_idx]
    source_targets = source_targets[valid_idx]
    target_targets = target_targets[valid_idx]
    
    return features, source_targets, target_targets

# Create tasks
features, source_targets, target_targets = create_tasks(data)

# Split data
split_idx = int(0.8 * len(features))

# Create datasets
source_train = torch.utils.data.TensorDataset(
    torch.FloatTensor(features[:split_idx]),
    torch.FloatTensor(source_targets[:split_idx])
)
source_val = torch.utils.data.TensorDataset(
    torch.FloatTensor(features[split_idx:]),
    torch.FloatTensor(source_targets[split_idx:])
)

target_train = torch.utils.data.TensorDataset(
    torch.FloatTensor(features[:split_idx]),
    torch.FloatTensor(target_targets[:split_idx])
)
target_val = torch.utils.data.TensorDataset(
    torch.FloatTensor(features[split_idx:]),
    torch.FloatTensor(target_targets[split_idx:])
)

# Create data loaders
source_train_loader = torch.utils.data.DataLoader(
    source_train, batch_size=64, shuffle=True
)
source_val_loader = torch.utils.data.DataLoader(
    source_val, batch_size=64
)
target_train_loader = torch.utils.data.DataLoader(
    target_train, batch_size=64, shuffle=True
)
target_val_loader = torch.utils.data.DataLoader(
    target_val, batch_size=64
)

## Model Training

Train the transfer learning model.

In [ ]:
# Initialize model
model = TransferModel(
    config=config,
    source_task="price_prediction",
    target_task="volatility_prediction",
    transfer_config=transfer_config
)

# Train model with transfer learning
history = model.transfer_learn(
    source_data=source_train_loader,
    target_data=target_train_loader,
    num_epochs=50,
    fine_tune=True,
    fine_tune_epochs=20
)

## Performance Analysis

Analyze the model's performance on both tasks.

In [ ]:
def plot_training_history(history):
    """Plot training history."""
    plt.figure(figsize=(15, 5))
    
    # Source task training
    plt.subplot(1, 3, 1)
    plt.plot(history['source_loss'])
    plt.title('Source Task Training')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    
    # Target task training
    plt.subplot(1, 3, 2)
    plt.plot(history['target_loss'])
    plt.title('Target Task Training')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    
    # Fine-tuning
    plt.subplot(1, 3, 3)
    plt.plot(history['fine_tune_loss'])
    plt.title('Fine-tuning')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    
    plt.tight_layout()
    plt.show()

# Plot training history
plot_training_history(history)

## Prediction Analysis

Analyze predictions on validation data.

In [ ]:
def analyze_predictions(model, source_loader, target_loader):
    """Analyze predictions on both tasks."""
    model.eval()
    source_preds = []
    source_true = []
    target_preds = []
    target_true = []
    
    with torch.no_grad():
        # Source task predictions
        for features, targets in source_loader:
            predictions = model(features, task="price_prediction")
            source_preds.extend(predictions.numpy())
            source_true.extend(targets.numpy())
        
        # Target task predictions
        for features, targets in target_loader:
            predictions = model(features, task="volatility_prediction")
            target_preds.extend(predictions.numpy())
            target_true.extend(targets.numpy())
    
    # Plot results
    plt.figure(figsize=(12, 5))
    
    # Price predictions
    plt.subplot(1, 2, 1)
    plt.scatter(source_true, source_preds, alpha=0.5)
    plt.plot([-1, 1], [-1, 1], 'r--')
    plt.title('Price Predictions')
    plt.xlabel('True Returns')
    plt.ylabel('Predicted Returns')
    
    # Volatility predictions
    plt.subplot(1, 2, 2)
    plt.scatter(target_true, target_preds, alpha=0.5)
    plt.plot([0, 0.1], [0, 0.1], 'r--')
    plt.title('Volatility Predictions')
    plt.xlabel('True Volatility')
    plt.ylabel('Predicted Volatility')
    
    plt.tight_layout()
    plt.show()

# Analyze predictions
analyze_predictions(model, source_val_loader, target_val_loader)

## Layer Analysis

Analyze the learned representations in shared layers.

In [ ]:
def analyze_layer_gradients(model):
    """Analyze gradients in shared layers."""
    gradients = model.get_layer_gradients()
    
    plt.figure(figsize=(10, 5))
    
    for i, (layer_name, grad) in enumerate(gradients.items()):
        plt.subplot(1, len(gradients), i+1)
        plt.hist(grad.numpy().flatten(), bins=50)
        plt.title(f'Layer {i+1} Gradients')
        plt.xlabel('Gradient Value')
        plt.ylabel('Count')
    
    plt.tight_layout()
    plt.show()

# Analyze layer gradients
analyze_layer_gradients(model)

## Save Model

Save the trained transfer learning model for future use.

In [ ]:
# Save model
save_path = 'models/transfer_model.pt'
model.save_transfer_model(save_path)
print(f"Model saved to {save_path}")